Feature weights test

In [94]:
# Sklearn imports
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV


# DiCE imports
import dice_ml
from dice_ml.utils import helpers  # helper functions

# pandas and numpy
import pandas as pd
import numpy as np

In [95]:
# Dataset from https://interpret.ml/DiCE/notebooks/DiCE_getting_started.html
# This dataset has 8 features.The outcome is income which is binarized to 0 (low-income, <=50K) or 1 (high-income, >50K).
df = helpers.load_adult_income_dataset()

print(df.head())

   age      workclass     education marital_status    occupation   race  \
0   28        Private     Bachelors         Single  White-Collar  White   
1   30  Self-Employed         Assoc        Married  Professional  White   
2   32        Private  Some-college        Married  White-Collar  White   
3   20        Private  Some-college         Single       Service  White   
4   41  Self-Employed  Some-college        Married  White-Collar  White   

   gender  hours_per_week  income  
0  Female              60       0  
1    Male              65       1  
2    Male              50       0  
3  Female              35       0  
4    Male              50       0  


In [96]:
# Split the dataset into train and test sets.
target = df["income"]
train_dataset, test_dataset, y_train, y_test = train_test_split(df,
                                                                target,
                                                                test_size=0.2,
                                                                random_state=0,
                                                                stratify=target)
x_train = train_dataset.drop('income', axis=1)
x_test = test_dataset.drop('income', axis=1)

In [97]:
#Given the train dataset, we construct a data object for DiCE. 
# Since continuous and discrete features have different ways of perturbation, we need to specify the names of the continuous features. 
# DiCE also requires the name of the output variable that the ML model will predict.
# Step 1: dice_ml.Data
d = dice_ml.Data(dataframe=train_dataset, continuous_features=['age', 'hours_per_week'], outcome_name='income')

In [98]:
# Gradient boosting classifier

numerical = [f'num_feature_{i}' for i in range(1, 11)]
categorical = x_train.columns.difference(numerical)

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical)])

# Append classifier to preprocessing pipeline.
# Now we have a full prediction pipeline.
clf = Pipeline(steps=[('preprocessor', transformations),
                      ('classifier', GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=42))])
mimodel = clf.fit(x_train, y_train)

In [99]:
# query instance in the form of a dictionary; keys: feature name, values: feature value
query_instance = {'age': 22,
                  'workclass': 'Private',
                  'education': 'HS-grad',
                  'marital_status': 'Single',
                  'occupation': 'Service',
                  'race': 'White',
                  'gender': 'Female',
                  'hours_per_week': 25}

query_df = pd.DataFrame([query_instance])
# Asegurarse de que las columnas del DataFrame coincidan con las de x_test
query_df = query_df[x_test.columns]

# Mostrar el DataFrame resultante
print(query_df)

   age workclass education marital_status occupation   race  gender  \
0   22   Private   HS-grad         Single    Service  White  Female   

   hours_per_week  
0              25  


In [100]:
#We now initialize the DiCE explainer, which needs a dataset and a model. DiCE provides local explanation for the model m and requires an query input whose outcome needs to be explained.
# Using sklearn backend
m = dice_ml.Model(model=mimodel, backend="sklearn")
exp2= dice_ml.Dice(d, m, method="genetic")

In [101]:
#create some CFS
e1 = exp2.generate_counterfactuals(query_df, total_CFs=10, desired_class="opposite")
e1.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  1.86it/s]

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,22,Private,HS-grad,Single,Service,White,Female,25,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,31,-,Prof-school,Married,Professional,-,Male,-,1
0,39,-,Masters,Married,Professional,-,-,24,1
0,17,-,Doctorate,Married,White-Collar,-,-,35,1
0,36,-,Bachelors,Married,Professional,-,-,32,1
0,31,-,Bachelors,Married,White-Collar,-,-,35,1
0,31,-,Bachelors,Married,Professional,-,-,35,1
0,38,-,Bachelors,Married,Professional,-,-,32,1
0,34,-,Bachelors,Married,Professional,-,-,35,1
0,30,-,Bachelors,Married,Professional,-,-,38,1
0,36,-,Bachelors,Married,Sales,-,-,35,1


In [103]:
# Making age more difficult to change, and gender and race imposible to change

feature_weights = {'age': 1000, 
                   'hours_per_week': 1, 
                   'workclass': 1,
                   'education': 1,
                   'marital_status': 1,
                   'occupation':  1,
                   'race':  1,
                   'gender': 1}
dice_exp2 = exp2.generate_counterfactuals(query_df, total_CFs=10, desired_class="opposite", feature_weights=feature_weights, permitted_range={'age': [22, 100]}, features_to_vary=['age', 'hours_per_week','workclass','education', 'marital_status','occupation'])
dice_exp2.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:09<00:00,  9.97s/it]

Query instance (original outcome : 0)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,22,Private,HS-grad,Single,Service,White,Female,25,0



Diverse Counterfactual set (new outcome: 1)


,age,workclass,education,marital_status,occupation,race,gender,hours_per_week,income
0,-,-,Prof-school,Married,White-Collar,-,-,40,1
0,-,-,Doctorate,Married,Professional,-,-,40,1
0,-,-,Doctorate,Married,White-Collar,-,-,1,1
0,26,-,Prof-school,Married,-,-,-,44,1
0,31,-,Prof-school,Married,Professional,-,-,-,1
0,-,Self-Employed,Doctorate,Married,Professional,-,-,40,1
0,-,Government,Doctorate,Married,Professional,-,-,40,1
0,-,Government,Prof-school,Married,White-Collar,-,-,40,1
